In [1]:
from vllm import LLM, SamplingParams
import pandas as pd
import numpy as np

def get_perplexity(token_list):
    """
        Calcula perplexity para una instancia única.
    """
    t = len(token_list)
    avg = (-1/t) * np.sum(token_list)
    perplexity = np.exp(avg)
    return round(perplexity, 4)

train = r'/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl'
dataset = pd.read_json(train, lines = True)

trans_prompt = """
    Given a set of premises, the task is to parse the problem and the question into first-order logic formulars. Answer only with the translated premises.
    The grammar of the first-order logic formular is defined as follows:
    1) logical conjunction of expr1 and expr2: expr1 ∧ expr2
    2) logical disjunction of expr1 and expr2: expr1 ∨ expr2
    3) logical exclusive disjunction of expr1 and expr2: expr1 ⊕ expr2
    4) logical negation of expr1: ¬expr1
    5) expr1 implies expr2: expr1 → expr2
    6) expr1 if and only if expr2: expr1 ↔ expr2
    7) logical universal quantification: ∀x
    8) logical existential quantification: ∃x
    --------------
    Natural Language Premises:
    All people who regularly drink coffee are dependent on caffeine. People either regularly drink coffee or joke about being addicted to caffeine. No one who jokes about being addicted to caffeine is unaware that caffeine is a drug. Rina is either a student and unaware that caffeine is a drug, or neither a student nor unaware that caffeine is a drug. If Rina is not a person dependent on caffeine and a student, then Rina is either a person dependent on caffeine and a student, or neither a person dependent on caffeine nor a student.
    Predicates:
    Dependent(x) ::: x is a person dependent on caffeine.
    Drinks(x) ::: x regularly drinks coffee.
    Jokes(x) ::: x jokes about being addicted to caffeine.
    Unaware(x) ::: x is unaware that caffeine is a drug.
    Student(x) ::: x is a student.
    FOL Premises:
    ∀x (Drinks(x) → Dependent(x)) ::: All people who regularly drink coffee are dependent on caffeine.
    ∀x (Drinks(x) ⊕ Jokes(x)) ::: People either regularly drink coffee or joke about being addicted to caffeine.
    ∀x (Jokes(x) → ¬Unaware(x)) ::: No one who jokes about being addicted to caffeine is unaware that caffeine is a drug. 
    (Student(rina) ∧ Unaware(rina)) ⊕ ¬(Student(rina) ∨ Unaware(rina)) ::: Rina is either a student and unaware that caffeine is a drug, or neither a student nor unaware that caffeine is a drug. 
    ¬(Dependent(rina) ∧ Student(rina)) → (Dependent(rina) ∧ Student(rina)) ⊕ ¬(Dependent(rina) ∨ Student(rina)) ::: If Rina is not a person dependent on caffeine and a student, then Rina is either a person dependent on caffeine and a student, or neither a person dependent on caffeine nor a student.
    --------------
    
    Natural Language Premises:
    {}
    FOL Premises:
"""

infer_prompt = """
    Given a set of premises and conclusion in first order logic, your task is to determine the logical validity of the conclusion: True, False, or Uncertain. Answer only with the logical value.
    A True conclusion is one that can be obtained via a valid inference procedure from the given premises.
    A False conclusion is one that contradicts one or more premises during the inference procedure. 
    An Uncertain conclusion is neither True nor False. Meaning that there is insufficient information in the premises to infer it, but the conclusion it self doesn't contradict any premise.
    --------------
    The following example shows a set of premises and conclusions where each conclusion represents a different logical validity. You should answer similarly.
    FOL-PREMISES:
    ∀x (WorkAt(x, meta) → HighIncome(x))
    ∀x (HighIncome(x) → ¬MeansToDestination(x, bus))
    ∀x (MeansToDestination(x, bus) ⊕ MeansToDestination(x, drive))
    ∀x (HaveCar(x) → MeansToDestination(x, drive))
    ∀x (Student(x) → ¬ MeansToDestination(x, drive))
    HaveCar(james) ∨ WorkAt(james, meta)
    --------------
    FOL-CONCLUSION:
    MeansToDestination(x, drive) ∨ Student(james)
    Student(james)
    ¬HighIncome(james)

    Analysis:
    The first conclusion is True. Premise 6 states that either James has a car (in which case premise 4 gives us the conclusion) or James works at Meta (in which case premise 4 implies premise 2, which combined with premise 3 gives us the conclusion)
    The second conclusion is False. Premise 5 states that students can't have a Car as a MeansToDestination, however the first condition tells us James has such means.
    The third conclusion is Uncertain. Premise 1 is the only guarantee to have a High Income, however we can't determine that James works at Meta (Premise 6).
    ----------------------------
    FOL-PREMISES:
    {}
    --------------
    FOL-CONCLUSION:
    {}
    --------------
    ANSWER:
"""


retrans_prompt = """
    Given a single premise in first order logic, your task is to translate the premise into natural language. Answer only with the translated premise. It should be a single sentence.
    The grammar of the first-order logic formular is defined as follows:
    1) logical conjunction of expr1 and expr2: expr1 ∧ expr2
    2) logical disjunction of expr1 and expr2: expr1 ∨ expr2
    3) logical exclusive disjunction of expr1 and expr2: expr1 ⊕ expr2
    4) logical negation of expr1: ¬expr1
    5) expr1 implies expr2: expr1 → expr2
    6) expr1 if and only if expr2: expr1 ↔ expr2
    7) logical universal quantification: ∀x
    8) logical existential quantification: ∃x
    --------------
    Below are examples of the translation:
    PREMISES:
    ¬(PartTime(jackie) ⊕ ForbesList(jackie)) → ∃y (LessThan(y, num2) ∧ TakesCourses(x,y)) ∧ ForbesList(jackie)
    ¬In(borjMasouda, tunisia)

    NATURAL LANGUAGE:
    If Jackie either enrolls as part-time in the current semester and is listed in the Forbes 30 Under 30, or neither enrolls as part-time in the current semester nor is listed in the Forbes 30 Under 30, then Jackie takes less than two courses in the current semester and listed in the Forbes 30 Under 30.
    Borj Masouda is not in Tunisia.
    --------------    
    PREMISE:
    {}

    NATURAL LANGUAGE:
"""

In [ ]:
def vllm_generation(model_id, quant):
    """
        Generación iterativa sobre distintos modelos usando vLLM.
    """
    print(f"Iniciando con: {model_id}...")
    
    trans_prompts = [trans_prompt.format(dataset['premises'][i]) for i in range(len(dataset['premises']))]
    infer_prompts = [infer_prompt.format(dataset['premises-FOL'][i], dataset['conclusion-FOL'][i]) for i in range(len(dataset['premises-FOL']))]
    retrans_prompts = [retrans_prompt.format(dataset['conclusion-FOL'][i]) for i in range(len(dataset['conclusion-FOL']))]

    prompts = trans_prompts + infer_prompts + retrans_prompts

    sampling_params = SamplingParams(temperature = 0.2, top_p = 0.9, max_tokens = 7000, seed = 67, logprobs = 1)
    llm = LLM(
        model = model_id, 
        max_model_len = 9000, 
        quantization = quant, 
        enforce_eager = True, 
        gpu_memory_utilization = 0.9, 
        limit_mm_per_prompt={"image": 0, "video": 0}
    )
    outputs = llm.generate(prompts, sampling_params)

    step_list = []
    perp_list = []
    i = 0
    for output in outputs:
        gen_text = output.outputs[0].text.strip(r'\n')
        step_list.append(gen_text)
        loglist = output.outputs[0].logprobs
        logit_values = [loglist[i].get(list(loglist[i].keys())[0]).logprob for i in range(len(loglist))]
        perp_list.append(get_perplexity(logit_values))
        if i % 20 == 0:
            print('Iteración: {}'.format(i))
        i += 1

        
    # To do: Pensar en otras cosas que medirle a las generaciones de los modelos.
    df_name = model_id.split('/')[1] # AHHHHHHHHHHHHHHHHHHHHHHH
    full_dataframe = pd.DataFrame({f"Full": step_list, "Perplexity": perp_list})
    full_dataframe.to_csv(f'/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{df_name}_full.csv')
    print("=" * 60)
    print("=" * 15 +  f" Fin de: {model_id} " + "=" * 15)
    print("=" * 60)


# 27B-32B no funciona papus :v Se queda al límite de la memoria. Usa 31.28GBs y Necesita 31.6GBs. Hay que jugarle con gpu_offloading, pero será para otro paper.
checkpoint_list = [
    #['Qwen/Qwen3-4B-FP8', 'fp8'], # Funciona bien. DONE
    #['Qwen/Qwen3-8B-FP8', 'fp8'], # Funciona bien. DONE
    #['Qwen/Qwen3-14B-FP8', 'fp8'], # Fuinciona bien. DONE
    ['Qwen/Qwen3.5-4B', 'fp8'], # Funciona bien. # No hay versión oficial quantizada
    ['Qwen/Qwen3.5-9B', 'fp8'], # Funciona bien. # No hay versión oficial quantizada
    ['google/gemma-3-4b-it','fp8'], # Funciona bien.
    ['google/gemma-3-12b-it','fp8'], # Funciona bien.
    ['openai/gpt-oss-20b', 'mxfp4'], # Funciona bien
    ['deepseek-ai/DeepSeek-R1-Distill-Qwen-14B', None], #Jala bien pero con gpu_memory_utilization = 0.95
    ['deepseek-ai/DeepSeek-R1-Distill-Llama-8B', None] # Jala bien.
]

# checkpoints que no corren: 
#[
# ['Qwen/Qwen3.5-27B-FP8', 'fp8'], 
# ['google/gemma-4-E2B', 'fp8'], 
# ['google/gemma-4-E4B', 'fp8'],
# ['google/gemma-4-26B-A4B','fp8'],
# ['google/gemma-3-27b-it','fp8'], 
#]

vllm_generation(checkpoint_list[0][0], checkpoint_list[0][1])

Iniciando con: Qwen/Qwen3-14B-FP8...
INFO 04-08 14:16:43 [utils.py:233] non-default args: {'max_model_len': 9000, 'disable_log_stats': True, 'quantization': 'fp8', 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 0, 'video': 0}, 'model': 'Qwen/Qwen3-14B-FP8'}
INFO 04-08 14:18:05 [model.py:533] Resolved architecture: Qwen3ForCausalLM
INFO 04-08 14:18:05 [model.py:1582] Using max model len 9000
INFO 04-08 14:18:05 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-08 14:18:05 [vllm.py:754] Asynchronous scheduling is enabled.
WARNING 04-08 14:18:05 [vllm.py:788] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 04-08 14:18:05 [vllm.py:799] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 04-08 14:18:05 [vllm.py:964] Cudagraph is disabled under eager mode
INFO 04-08 1

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(EngineCore pid=1525103) INFO 04-08 14:19:43 [default_loader.py:384] Loading weights took 14.70 seconds
(EngineCore pid=1525103) INFO 04-08 14:19:43 [gpu_model_runner.py:4566] Model loading took 15.33 GiB memory and 95.785462 seconds
(EngineCore pid=1525103) WARNING 04-08 14:19:44 [fp8_utils.py:1174] Using default W8A8 Block FP8 kernel config. Performance might be sub-optimal! Config file not found at /home/flopezp/.venvs/foo/lib/python3.13/site-packages/vllm/model_executor/layers/quantization/utils/configs/N=7168,K=5120,device_name=NVIDIA_RTX_5000_Ada_Generation,dtype=fp8_w8a8,block_shape=[128,128].json
(EngineCore pid=1525103) WARNING 04-08 14:19:44 [fp8_utils.py:1174] Using default W8A8 Block FP8 kernel config. Performance might be sub-optimal! Config file not found at /home/flopezp/.venvs/foo/lib/python3.13/site-packages/vllm/model_executor/layers/quantization/utils/configs/N=5120,K=5120,device_name=NVIDIA_RTX_5000_Ada_Generation,dtype=fp8_w8a8,block_shape=[128,128].json
(EngineCor

Rendering prompts:   0%|          | 0/678 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/678 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=1525103) INFO 04-08 16:08:08 [core.py:1201] Shutdown initiated (timeout=0)
(EngineCore pid=1525103) INFO 04-08 16:08:08 [core.py:1224] Shutdown complete
Iteración: 0
Iteración: 20
Iteración: 40
Iteración: 60
Iteración: 80
Iteración: 100
Iteración: 120
Iteración: 140
Iteración: 160
Iteración: 180
Iteración: 200
Iteración: 220
Iteración: 240
Iteración: 260
Iteración: 280
Iteración: 300
Iteración: 320
Iteración: 340
Iteración: 360
Iteración: 380
Iteración: 400
Iteración: 420
Iteración: 440
Iteración: 460
Iteración: 480
Iteración: 500
Iteración: 520
Iteración: 540
Iteración: 560
Iteración: 580
Iteración: 600
Iteración: 620
Iteración: 640
Iteración: 660
=============== Fin de: Qwen/Qwen3-14B-FP8 ===============


**Pruebas**

Lo que sigue es jalar los distintos checkpoints para ver que funcionen

In [1]:
from vllm import LLM, SamplingParams
from huggingface_hub import login

prompts = [
        "Who are you?",
        "What is AI?"
    ]
sampling_params = SamplingParams(temperature = 0.2, top_p = 0.9, max_tokens = 70, seed = 67, logprobs = 1)
llm = LLM(
    model = 'Qwen/Qwen3.5-27B-FP8', 
    max_model_len = 100, 
    quantization = None, 
    enforce_eager = True, 
    trust_remote_code = True, 
    gpu_memory_utilization = 0.9,
    cpu_offload_gb = 4, # Pruebas para los modelos de más de 20B
    limit_mm_per_prompt={"image": 0, "video": 0}
)


outputs = llm.generate(prompts, sampling_params)
for output in outputs:
        prompt = output.prompt
        gen_text = output.outputs[0].text.strip(r'\n')
        print(f"Prompt: {prompt!r}")
        print(f"Answer: {gen_text!r}")
        print("-" * 15)

INFO 04-07 15:25:48 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 100, 'cpu_offload_gb': 4, 'disable_log_stats': True, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 0, 'video': 0}, 'model': 'Qwen/Qwen3.5-27B-FP8'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
`layer_type_validation` is deprecated and will be removed in v5.20. Use `PreTrainedConfig.validate_layer_type` instead
`layer_type_validation` is deprecated and will be removed in v5.20. Use `PreTrainedConfig.validate_layer_type` instead
`layer_type_validation` is deprecated and will be removed in v5.20. Use `PreTrainedConfig.validate_layer_type` instead


INFO 04-07 15:27:09 [model.py:533] Resolved architecture: Qwen3_5ForConditionalGeneration
INFO 04-07 15:27:09 [model.py:1582] Using max model len 100
INFO 04-07 15:27:09 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.


`Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.


INFO 04-07 15:27:09 [config.py:212] Setting attention block size to 784 tokens to ensure that attention page size is >= mamba page size.
INFO 04-07 15:27:09 [config.py:243] Padding mamba page size by 0.13% to ensure that mamba page size and attention page size are exactly equal.
INFO 04-07 15:27:09 [vllm.py:754] Asynchronous scheduling is enabled.
WARNING 04-07 15:27:09 [vllm.py:788] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 04-07 15:27:09 [vllm.py:799] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 04-07 15:27:09 [vllm.py:964] Cudagraph is disabled under eager mode
INFO 04-07 15:27:09 [compilation.py:289] Enabled custom fusions: norm_quant, act_quant
INFO 04-07 15:27:12 [registry.py:126] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
WARNING 04

...

(EngineCore pid=699658) INFO 04-07 16:05:38 [default_loader.py:384] Loading weights took 3.95 seconds
(EngineCore pid=699658) INFO 04-07 16:05:40 [gpu_model_runner.py:4566] Model loading took 23.62 GiB memory and 10.320286 seconds
(EngineCore pid=699658) WARNING 04-07 16:05:41 [fp8_utils.py:1174] Using default W8A8 Block FP8 kernel config. Performance might be sub-optimal! Config file not found at /home/flopezp/.venvs/foo/lib/python3.13/site-packages/vllm/model_executor/layers/quantization/utils/configs/N=16384,K=5120,device_name=NVIDIA_RTX_5000_Ada_Generation,dtype=fp8_w8a8,block_shape=[128,128].json


(EngineCore pid=699658) /home/flopezp/.venvs/foo/lib/python3.13/site-packages/vllm/model_executor/layers/fla/ops/utils.py:113: UserWarning: Input tensor shape suggests potential format mismatch: seq_len (16) < num_heads (48). This may indicate the inputs were passed in head-first format [B, H, T, ...] when head_first=False was specified. Please verify your input tensor format matches the expected shape [B, T, H, ...].
(EngineCore pid=699658)   return fn(*contiguous_args, **contiguous_kwargs)
(EngineCore pid=699658) /home/flopezp/.venvs/foo/lib/python3.13/site-packages/vllm/model_executor/layers/fla/ops/utils.py:113: UserWarning: Input tensor shape suggests potential format mismatch: seq_len (32) < num_heads (48). This may indicate the inputs were passed in head-first format [B, H, T, ...] when head_first=False was specified. Please verify your input tensor format matches the expected shape [B, T, H, ...].
(EngineCore pid=699658)   return fn(*contiguous_args, **contiguous_kwargs)


(EngineCore pid=699658) WARNING 04-07 16:05:59 [fp8_utils.py:1174] Using default W8A8 Block FP8 kernel config. Performance might be sub-optimal! Config file not found at /home/flopezp/.venvs/foo/lib/python3.13/site-packages/vllm/model_executor/layers/quantization/utils/configs/N=5120,K=6144,device_name=NVIDIA_RTX_5000_Ada_Generation,dtype=fp8_w8a8,block_shape=[128,128].json
(EngineCore pid=699658) WARNING 04-07 16:05:59 [fp8_utils.py:1174] Using default W8A8 Block FP8 kernel config. Performance might be sub-optimal! Config file not found at /home/flopezp/.venvs/foo/lib/python3.13/site-packages/vllm/model_executor/layers/quantization/utils/configs/N=34816,K=5120,device_name=NVIDIA_RTX_5000_Ada_Generation,dtype=fp8_w8a8,block_shape=[128,128].json
(EngineCore pid=699658) WARNING 04-07 16:06:00 [fp8_utils.py:1174] Using default W8A8 Block FP8 kernel config. Performance might be sub-optimal! Config file not found at /home/flopezp/.venvs/foo/lib/python3.13/site-packages/vllm/model_executor/l

(EngineCore pid=699658) Process EngineCore:
(EngineCore pid=699658) Traceback (most recent call last):
(EngineCore pid=699658)   File "/usr/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
(EngineCore pid=699658)     self.run()
(EngineCore pid=699658)     ~~~~~~~~^^
(EngineCore pid=699658)   File "/usr/lib/python3.13/multiprocessing/process.py", line 108, in run
(EngineCore pid=699658)     self._target(*self._args, **self._kwargs)
(EngineCore pid=699658)     ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=699658)   File "/home/flopezp/.venvs/foo/lib/python3.13/site-packages/vllm/v1/engine/core.py", line 1103, in run_engine_core
(EngineCore pid=699658)     raise e
(EngineCore pid=699658)   File "/home/flopezp/.venvs/foo/lib/python3.13/site-packages/vllm/v1/engine/core.py", line 1073, in run_engine_core
(EngineCore pid=699658)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=699658)   File "/home/flopezp/.venvs/foo/lib/p

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}